# EDA — Previsão de Satisfação do Cliente (Olist)

## Contexto e Domínio

Dataset público da [Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) com dados reais e anonimizados de compras realizadas em marketplaces brasileiros entre **2016 e 2018**.

## Problema de Negócio

**Prever a nota de satisfação do cliente** após uma compra, para que a empresa consiga identificar pedidos com maior risco de gerar avaliações ruins e atuar preventivamente — melhorando logística, atendimento e reputação dos vendedores.

## Variável-Alvo

- **Coluna:** `review_score`  
- **Classes:** 1, 2, 3, 4 e 5 estrelas  
- **Tipo de problema:** classificação multiclasse (5 classes)

## Critério de Sucesso

O modelo seria útil se conseguisse **antecipar pedidos com alta chance de receber 1 ou 2 estrelas**, permitindo ação preventiva: contato proativo com o cliente, priorização de suporte ou correção de problemas de entrega.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## 1. Carregamento dos Dados

O dataset é composto por 8 arquivos CSV cobrindo pedidos, clientes, produtos, pagamentos e avaliações.

In [ ]:
DATA_PATH = "data/"

orders     = pd.read_csv(DATA_PATH + "olist_orders_dataset.csv")
customers  = pd.read_csv(DATA_PATH + "olist_customers_dataset.csv")
items      = pd.read_csv(DATA_PATH + "olist_order_items_dataset.csv")
payments   = pd.read_csv(DATA_PATH + "olist_order_payments_dataset.csv")
reviews    = pd.read_csv(DATA_PATH + "olist_order_reviews_dataset.csv")
products   = pd.read_csv(DATA_PATH + "olist_products_dataset.csv")
sellers    = pd.read_csv(DATA_PATH + "olist_sellers_dataset.csv")
categories = pd.read_csv(DATA_PATH + "product_category_name_translation.csv")

In [ ]:
datasets = {
    "orders": orders, "customers": customers, "items": items,
    "payments": payments, "reviews": reviews,
    "products": products, "sellers": sellers, "categories": categories
}

for name, df_aux in datasets.items():
    print(f"{name:12s} → {df_aux.shape[0]:7,} linhas  ×  {df_aux.shape[1]:2} colunas")

## 2. Visão Geral das Tabelas Principais

In [ ]:
orders.head()

In [ ]:
items.head()

In [ ]:
payments.head()

In [ ]:
reviews.head()

## 3. Construção do Dataset Principal

Unimos as principais tabelas em um único DataFrame onde **cada linha representa um pedido**.

Além das features brutas, criamos duas variáveis derivadas com potencial preditivo:
- **`delivery_time_days`**: dias entre compra e entrega efetiva
- **`delivery_late`**: se o pedido chegou após a data estimada (1 = atrasado)

In [ ]:
items_agg = items.groupby("order_id").agg(
    n_items=("order_item_id", "count"),
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum")
).reset_index()

payments_agg = payments.groupby("order_id").agg(
    payment_value=("payment_value", "sum"),
    payment_type=("payment_type", lambda x: x.mode()[0]),
    payment_installments=("payment_installments", "max")
).reset_index()

# Para o target, usamos o primeiro review registrado por pedido
reviews_target = reviews.groupby("order_id")["review_score"].first().reset_index()

df = (
    orders
    .merge(customers[["customer_id", "customer_state"]], on="customer_id", how="left")
    .merge(items_agg, on="order_id", how="left")
    .merge(payments_agg, on="order_id", how="left")
    .merge(reviews_target, on="order_id", how="left")
)

print("Shape após merge:", df.shape)

In [ ]:
# Converte timestamps
date_cols = ["order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"]
for col in date_cols:
    df[col] = pd.to_datetime(df[col])

# Features derivadas de logística
df["delivery_time_days"] = (
    df["order_delivered_customer_date"] - df["order_purchase_timestamp"]
).dt.days

df["delivery_late"] = (
    (df["order_delivered_customer_date"] > df["order_estimated_delivery_date"]) &
    df["order_delivered_customer_date"].notna()
).astype(int)

# Mantemos apenas pedidos com avaliação (target não pode ser nulo)
df = df.dropna(subset=["review_score"])
df["review_score"] = df["review_score"].astype(int)

print("Shape final (com review):", df.shape)

In [ ]:
df.info()

## 4. Train/Test Split — A Regra de Ouro

Antes de qualquer análise, separamos os dados em treino e teste para **evitar vazamento de dados** (*data leakage*).  
Toda a EDA a seguir será feita **exclusivamente com `X_train` e `y_train`**.

In [ ]:
drop_cols = [
    "order_id", "customer_id",
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "review_score"  # target
]

X = df.drop(columns=drop_cols)
y = df["review_score"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"\nDistribuição do target em y_train:")
print(y_train.value_counts().sort_index())

In [ ]:
# DataFrame de treino completo (features + target) para uso nas análises
train_df = X_train.copy()
train_df["review_score"] = y_train.values

## 5. Distribuição da Variável Alvo (`review_score`)

Antes de explorar as features, entendemos como o target está distribuído. Isso é fundamental para identificar desbalanceamento de classes, que impacta diretamente a escolha do modelo e a métrica de avaliação.

In [ ]:
score_counts = y_train.value_counts().sort_index()

plt.figure(figsize=(8, 5))
sns.barplot(x=score_counts.index, y=score_counts.values, palette="RdYlGn")
plt.title("Distribuição da Nota de Avaliação (y_train)")
plt.xlabel("Nota (review_score)")
plt.ylabel("Contagem")
plt.tight_layout()
plt.show()

In [ ]:
print("Proporção de cada classe em y_train:")
print((y_train.value_counts(normalize=True).sort_index() * 100).round(1).rename("% do total"))

## 6. Análise de Valores Ausentes

Identificamos e quantificamos os NaNs em `X_train`. Qualquer estratégia de imputação deve ser **aprendida apenas no treino** e aplicada ao teste.

In [ ]:
nan_counts = X_train.isnull().sum()
nan_pct    = (nan_counts / len(X_train) * 100).round(2)

nan_df = pd.DataFrame({"qtd_nan": nan_counts, "pct_nan (%)": nan_pct})
nan_df[nan_df["qtd_nan"] > 0].sort_values("pct_nan (%)", ascending=False)

In [ ]:
plt.figure(figsize=(14, 4))
sns.heatmap(X_train.isnull(), cbar=False, yticklabels=False, cmap="viridis")
plt.title("Padrão de Valores Ausentes — X_train")
plt.tight_layout()
plt.show()

## 7. Variáveis Numéricas

Estatísticas descritivas e distribuições. Note que `delivery_time_days` e `delivery_late` são features derivadas de logística — variáveis com potencial forte de predição da satisfação do cliente.

In [ ]:
num_cols = ["total_price", "total_freight", "payment_value",
            "payment_installments", "n_items", "delivery_time_days", "delivery_late"]
X_train[num_cols].describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    X_train[col].dropna().hist(ax=axes[i], bins=40, color="steelblue", edgecolor="white")
    axes[i].set_title(col)

axes[-1].set_visible(False)
plt.suptitle("Distribuição das Variáveis Numéricas (X_train)", fontsize=14)
plt.tight_layout()
plt.show()

## 8. Variáveis Categóricas

Distribuição das principais variáveis categóricas no conjunto de treino.

In [ ]:
status_counts = X_train["order_status"].value_counts()

plt.figure(figsize=(10, 5))
sns.barplot(x=status_counts.index, y=status_counts.values, palette="viridis")
plt.title("Distribuição de Status dos Pedidos")
plt.xlabel("Status")
plt.ylabel("Contagem")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(
    data=X_train,
    x="payment_type",
    order=X_train["payment_type"].value_counts().index,
    palette="viridis"
)
plt.title("Distribuição de Tipo de Pagamento")
plt.xlabel("Tipo de Pagamento")
plt.ylabel("Contagem")
plt.tight_layout()
plt.show()

In [ ]:
top_states = X_train["customer_state"].value_counts().head(10)

plt.figure(figsize=(10, 5))
sns.barplot(x=top_states.index, y=top_states.values, palette="viridis")
plt.title("Top 10 Estados por Número de Pedidos")
plt.xlabel("Estado")
plt.ylabel("Contagem")
plt.tight_layout()
plt.show()

In [ ]:
train_order_ids = df.loc[X_train.index, "order_id"]

train_items_cat = (
    items
    .merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
    .merge(categories, on="product_category_name", how="left")
)
train_items_cat = train_items_cat[train_items_cat["order_id"].isin(train_order_ids)]

top_cats = train_items_cat["product_category_name_english"].dropna().value_counts().head(10)

plt.figure(figsize=(12, 5))
sns.barplot(x=top_cats.values, y=top_cats.index, palette="viridis")
plt.title("Top 10 Categorias de Produtos (por nº de itens vendidos)")
plt.xlabel("Contagem")
plt.ylabel("Categoria")
plt.tight_layout()
plt.show()

## 9. Categorias vs Variável Alvo (`review_score`)

Investigamos se o tipo de pagamento e o estado do cliente influenciam a nota de avaliação. Usamos boxplot, pois `review_score` é ordinal (1 a 5).

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(
    data=train_df,
    x="payment_type",
    y="review_score",
    order=train_df["payment_type"].value_counts().index,
    palette="viridis"
)
plt.title("Nota de Avaliação por Tipo de Pagamento")
plt.xlabel("Tipo de Pagamento")
plt.ylabel("Nota de Avaliação")
plt.tight_layout()
plt.show()

In [ ]:
# Nota média por estado (Top 10 estados por volume)
top_10_states = X_train["customer_state"].value_counts().head(10).index
mean_score_state = (
    train_df[train_df["customer_state"].isin(top_10_states)]
    .groupby("customer_state")["review_score"]
    .mean()
    .sort_values()
)

plt.figure(figsize=(10, 5))
sns.barplot(x=mean_score_state.index, y=mean_score_state.values, palette="RdYlGn")
plt.title("Nota Média de Avaliação por Estado (Top 10 estados)")
plt.xlabel("Estado")
plt.ylabel("Nota Média")
plt.ylim(3, 5)
plt.tight_layout()
plt.show()

## 10. Variáveis de Logística vs Variável Alvo

As variáveis `delivery_late` e `delivery_time_days` são candidatas a features importantes. Aqui investigamos visualmente sua relação com `review_score`.

In [ ]:
# Nota média por entrega no prazo vs atrasada
plt.figure(figsize=(7, 5))
sns.boxplot(
    data=train_df,
    x="delivery_late",
    y="review_score",
    palette="coolwarm"
)
plt.title("Nota de Avaliação vs Atraso na Entrega")
plt.xlabel("Atrasado (0 = Não, 1 = Sim)")
plt.ylabel("Nota de Avaliação")
plt.tight_layout()
plt.show()

In [ ]:
# Tempo de entrega por nota: distribuição via boxplot
cap = train_df["delivery_time_days"].quantile(0.97)
plot_df = train_df[train_df["delivery_time_days"] <= cap].dropna(subset=["delivery_time_days"])

plt.figure(figsize=(10, 5))
sns.boxplot(
    data=plot_df,
    x="review_score",
    y="delivery_time_days",
    palette="RdYlGn"
)
plt.title("Tempo de Entrega (dias) por Nota de Avaliação")
plt.xlabel("Nota de Avaliação")
plt.ylabel("Tempo de Entrega (dias)")
plt.tight_layout()
plt.show()

## 11. Análise Temporal

Investigamos padrões de volume de pedidos ao longo do tempo.

In [ ]:
train_orders = df.loc[X_train.index, ["order_purchase_timestamp"]].copy()
train_orders["month"]       = train_orders["order_purchase_timestamp"].dt.to_period("M")
train_orders["day_of_week"] = train_orders["order_purchase_timestamp"].dt.day_name()
train_orders["hour"]        = train_orders["order_purchase_timestamp"].dt.hour

monthly = train_orders["month"].value_counts().sort_index()

plt.figure(figsize=(14, 5))
monthly.plot(kind="bar", color="steelblue", edgecolor="white")
plt.title("Pedidos por Mês")
plt.xlabel("Mês")
plt.ylabel("Nº de Pedidos")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
day_order  = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
dow_counts = train_orders["day_of_week"].value_counts().reindex(day_order)

plt.figure(figsize=(10, 5))
sns.barplot(x=dow_counts.index, y=dow_counts.values, palette="viridis")
plt.title("Pedidos por Dia da Semana")
plt.xlabel("Dia")
plt.ylabel("Nº de Pedidos")
plt.tight_layout()
plt.show()

In [ ]:
hour_counts = train_orders["hour"].value_counts().sort_index()

plt.figure(figsize=(12, 5))
sns.barplot(x=hour_counts.index, y=hour_counts.values, palette="viridis")
plt.title("Pedidos por Hora do Dia")
plt.xlabel("Hora")
plt.ylabel("Nº de Pedidos")
plt.tight_layout()
plt.show()

## 12. Correlações entre Variáveis Numéricas

Calculamos as correlações de Pearson entre as variáveis numéricas do treino, incluindo o target `review_score`.

In [ ]:
corr_features = ["total_price", "total_freight", "payment_value",
                 "payment_installments", "n_items", "delivery_time_days", "delivery_late"]

corr_df = X_train[corr_features].copy()
corr_df["review_score"] = y_train.values

plt.figure(figsize=(10, 8))
sns.heatmap(corr_df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Matriz de Correlação (X_train + review_score)")
plt.tight_layout()
plt.show()

## Conclusões

Principais observações desta EDA:

- **Desbalanceamento do target:** a nota 5 domina o dataset (~57% dos pedidos), enquanto notas 2 e 3 são as classes mais raras. Isso exige atenção na escolha da métrica — acurácia simples será enganosa; prefira *weighted F1* ou *macro F1*.
- **Logística como fator-chave:** pedidos atrasados (`delivery_late = 1`) recebem notas significativamente menores, e pedidos com maior `delivery_time_days` também tendem a ter avaliações piores. Essas serão features importantes para o modelo.
- **Valores ausentes:** `delivery_time_days` tem NaNs nos pedidos ainda não entregues (status != `delivered`). Estratégia sugerida: imputação com mediana ou flag de ausência.
- **Pagamentos:** cartão de crédito é o método dominante. Pedidos via boleto e voucher têm distribuições distintas de valor e potencialmente de satisfação.
- **Distribuição geográfica:** SP concentra o maior volume, mas a nota média varia entre estados — possível indicador de qualidade logística regional.
- **Crescimento do negócio:** há crescimento expressivo de pedidos em 2017-2018, com pico em novembro (Black Friday). Sazonalidade pode impactar tanto o volume quanto a satisfação.
- **Consideração ética:** o modelo não deve responsabilizar exclusivamente vendedores por notas baixas causadas por problemas logísticos ou de transportadora. Interpretabilidade e monitoramento de viés por categoria/região são essenciais em produção.